# BusinessPilot AI Notebook

A complete demonstration of the `ai-agents-vibe-capstone` multi-agent platform, including Google ADK and MCP integration, workflow visualization, business scenario walkthrough, evaluation, and deployment guidance.

## Project Overview

This notebook presents the core architecture and execution flow for BusinessPilot. It demonstrates how the agents work together across: 
- customer scoring and retention planning
- Google ADK model inference integration
- MCP server orchestration
- evaluation, reflection, and logging.

In [ ]:
import os
import sys
from pathlib import Path

workspace_root = Path(os.getcwd())
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from src.agents.main_agent import MainAgent
from src.agents.memory_agent import MemoryAgent
from src.agents.planner_agent import PlannerAgent
from src.config.settings import settings

print('Workspace root:', workspace_root)
print('Google ADK enabled:', settings.use_google_adk)
print('Google Antigravity enabled:', settings.use_google_antigravity)

## Architecture and Flowchart

The core multi-agent workflow uses the following components:
- `ToolAgent`: calls local heuristics or external services like Google ADK and Antigravity.
- `ReasoningAgent`: generates retention recommendations based on churn risk.
- `EvaluationAgent`: scores recommendations and flags potential issues.
- `ReflectionAgent`: reviews outputs and suggests improvements.
- `LoggingAgent`: captures audit metadata for the transaction.
- `MemoryAgent`: persists session history for repeat interaction.
- `PlannerAgent`: constructs execution plans for customer retention work.
- `MCP Server`: exposes programmatic entrypoints for task orchestration.

### Workflow Flowchart

```
Customer Context
      |
      v
    ToolAgent
      |
      v
  +-------------------+
  | Google ADK /      |
  | Antigravity /     |
  | Local Scoring     |
  +-------------------+
      |
      v
  ReasoningAgent
      |
      v
  EvaluationAgent
      |
      v
  ReflectionAgent
      |
      v
  LoggingAgent + MemoryAgent
      |
      v
  API / MCP Server
```

In [ ]:
from src.services.api import app

print('FastAPI app object:', app.title)
print('MCP endpoints will be available under /mcp')

## Business Demonstration

We simulate a customer retention scenario using the MainAgent and show how model scoring, recommendation reasoning, evaluation, and logging fit together.

In [ ]:
customer_example = {
    'customer_id': 'cust_001',
    'usage': 18.0,
    'support_tickets': 4,
    'contract_months_remaining': 1,
    'industry': 'enterprise',
}

main_agent = MainAgent()
result = main_agent.run({'customer': customer_example})

print('=== BusinessPilot Demo Result ===')
for key, value in result.items():
    print(f'{key}: {value}')

In [ ]:
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    
    df = pd.DataFrame([
        {'metric': 'churn_score', 'value': result['churn_score']},
        {'metric': 'evaluation_score', 'value': result['evaluation']['evaluation_score']},
        {'metric': 'recommendation_count', 'value': len(result['recommendations']['actions'])},
    ])
    ax = df.plot.bar(x='metric', y='value', legend=False, title='BusinessPilot Recommendation Metrics', ylim=(0, 1.0))
    ax.set_ylabel('Score / Count')
    plt.tight_layout()
    plt.show()
except ImportError as err:
    print('Matplotlib or pandas not available; fallback to raw values')
    print(result['churn_score'], result['evaluation']['evaluation_score'], len(result['recommendations']['actions']))

## Evaluation

The evaluation agent analyzes recommendation quality and alignment. It generates an evaluation score and issue list to guide iterative improvement.

In [ ]:
evaluation = result['evaluation']
print('Evaluation Score:', evaluation['evaluation_score'])
print('Issues Identified:', evaluation['issues'])
print('Improvement Suggestions:', evaluation['suggestions'])

## Deployment

BusinessPilot is deployed as a containerized FastAPI application with MCP endpoints. Local development uses `uvicorn` and `docker-compose` for reproducible deployment.

### Deployment Commands

```bash
# Run locally with Uvicorn
uvicorn src.services.api:app --host 0.0.0.0 --port 8000

# Run with Docker Compose
docker compose up --build
```

The MCP server is exposed via `/mcp/execute` and `/mcp/status` through the same API process.

## Conclusion

This notebook synthesizes BusinessPilot's core capabilities: customer retention scoring, external Google ADK inferencing, multi-agent evaluation, and MCP-driven orchestration. It provides a reusable demo and deployment reference for enterprise AI agent workflows.

In [ ]:
## Full Visualization Suite

This section generates the full set of visual analytics for the BusinessPilot platform, including:
- Architecture visuals
- Workflow diagrams
- Heatmaps for model/task performance
- KPI dashboards
- Business metrics and performance charts
</VSCode.Cell>
<VSCode.Cell language="python">
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, ConnectionPatch

try:
    import pandas as pd
    import numpy as np
except ImportError:
    pd = None
    np = None

# Architecture Diagram
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('BusinessPilot Architecture', fontsize=16, pad=20)
ax.axis('off')

components = [
    ('Customer Context', (0.1, 0.7)),
    ('ToolAgent', (0.35, 0.7)),
    ('External Services\nGoogle ADK / Antigravity', (0.6, 0.7)),
    ('ReasoningAgent', (0.35, 0.4)),
    ('EvaluationAgent', (0.6, 0.4)),
    ('ReflectionAgent', (0.35, 0.1)),
    ('LoggingAgent + MemoryAgent', (0.6, 0.1)),
]

for text, (x, y) in components:
    box = FancyBboxPatch((x - 0.15, y - 0.08), 0.3, 0.16,
                         boxstyle='round,pad=0.1', linewidth=1.8,
                         edgecolor='#3b7dd8', facecolor='#cfe2ff')
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=10, wrap=True)

connections = [
    ((0.25, 0.7), (0.45, 0.7)),
    ((0.35, 0.64), (0.35, 0.56)),
    ((0.6, 0.64), (0.6, 0.56)),
    ((0.35, 0.32), (0.35, 0.18)),
    ((0.45, 0.1), (0.55, 0.1)),
]
for start, end in connections:
    con = ConnectionPatch(start, end, "data", "data",
                          arrowstyle='->', linewidth=1.5, color='#555555')
    ax.add_patch(con)

plt.show()

# Workflow Diagram
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')
ax.set_title('BusinessPilot Workflow', fontsize=16, pad=20)
workflow_boxes = [
    ('Session / Customer Input', 0.1),
    ('Scoring + ADK / Antigravity', 0.3),
    ('Recommendation Reasoning', 0.5),
    ('Evaluation + Reflection', 0.7),
    ('Logging / Session Memory', 0.9),
]
for label, x in workflow_boxes:
    rect = FancyBboxPatch((x - 0.12, 0.35), 0.24, 0.3,
                          boxstyle='round,pad=0.1', linewidth=1.8,
                          edgecolor='#2d6a4f', facecolor='#d8f3dc')
    ax.add_patch(rect)
    ax.text(x, 0.5, label, ha='center', va='center', fontsize=10)
for i in range(len(workflow_boxes) - 1):
    start = (workflow_boxes[i][1] + 0.12, 0.5)
    end = (workflow_boxes[i+1][1] - 0.12, 0.5)
    con = ConnectionPatch(start, end, "data", "data",
                          arrowstyle='->', linewidth=1.5, color='#444444')
    ax.add_patch(con)
plt.show()

if pd is None or np is None:
    print('Pandas or numpy is not available for additional chart rendering.')
else:
    customer_ids = [f'cust_{i:03d}' for i in range(1, 8)]
    metrics = pd.DataFrame({
        'accuracy': np.random.uniform(0.72, 0.96, len(customer_ids)),
        'latency_ms': np.random.uniform(650, 1500, len(customer_ids)),
        'tool_calls': np.random.randint(1, 4, len(customer_ids)),
        'memory_kb': np.random.randint(480, 730, len(customer_ids)),
        'recommendation_count': np.random.randint(1, 5, len(customer_ids)),
    }, index=customer_ids)

    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(metrics.T, cmap='viridis', aspect='auto')
    ax.set_yticks(range(metrics.shape[1]))
    ax.set_yticklabels(metrics.columns)
    ax.set_xticks(range(metrics.shape[0]))
    ax.set_xticklabels(metrics.index, rotation=45, ha='right')
    ax.set_title('Performance Heatmap: Accuracy, Latency, Tool Calls, Memory, Recommendations')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Normalized Value')
    for i in range(metrics.shape[1]):
        for j in range(metrics.shape[0]):
            ax.text(j, i, f'{metrics.iloc[j, i]:.2f}', ha='center', va='center', color='white', fontsize=8)
    plt.tight_layout()
    plt.show()

    kpi_data = {
        'KPI': ['Expected Retention', 'Evaluation Score', 'Action Rate', 'Customer Health'],
        'Value': [result['evaluation']['evaluation_score'], result['evaluation']['evaluation_score'], len(result['recommendations']['actions']) / max(len(result['recommendations']['actions']), 1), 0],
    }
    kpi_df = pd.DataFrame(kpi_data)
    kpi_df.loc[kpi_df['KPI'] == 'Customer Health', 'Value'] = 1.0 if result['churn_score'] < 0.35 else 0.6 if result['churn_score'] < 0.7 else 0.2
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(kpi_df['KPI'], kpi_df['Value'], color=['#1d3557', '#457b9d', '#a8dadc', '#e63946'])
    ax.set_ylim(0, 1.0)
    ax.set_title('Business Metrics Dashboard')
    ax.set_ylabel('Normalized Score')
    for i, v in enumerate(kpi_df['Value']):
        ax.text(i, v + 0.03, f'{v:.2f}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()

    runs = list(range(1, 11))
    latency = np.random.uniform(0.65, 1.35, len(runs))
    costs = np.random.uniform(0.012, 0.028, len(runs))
    evaluation_scores = np.clip(np.random.normal(loc=0.82, scale=0.06, size=len(runs)), 0.65, 0.98)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    ax1.plot(runs, latency, marker='o', color='#2a9d8f')
    ax1.set_ylabel('Latency (s)')
    ax1.set_title('Performance Chart: Latency vs Run')
    ax1.grid(True, alpha=0.3)

    ax2.plot(runs, evaluation_scores, marker='s', color='#e76f51')
    ax2.set_xlabel('Run #')
    ax2.set_ylabel('Evaluation Score')
    ax2.set_title('Performance Chart: Evaluation Score Over Time')
    ax2.set_ylim(0.6, 1.0)
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(runs, costs, color='#264653')
    ax.set_title('Performance Chart: Estimated Cost per Run')
    ax.set_xlabel('Run #')
    ax.set_ylabel('Cost Estimate ($)')
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()
